# 14.11 · 因果性检验 / Granger Causality

> **课程定位 / Where this fits**
> 第 11 课，**Part 14 · 时间序列**(本部分收尾)。从"预测"转向"探究关系"。
> Lesson 11, **Part 14 · Time Series** (finale). From "forecasting" to "probing relationships."
>
> 预测关心的是**相关**——能预测就行。但有时我们想问更深的问题:"**X 是不是真的影响/能预测 Y?**"(广告投放影响销量吗? 利率影响通胀吗?) **Granger 因果检验**给出一种可操作的回答: 如果**加入 X 的历史能显著改善对 Y 的预测**(超过只用 Y 自己历史的效果), 就说 "X **Granger-导致** Y"。本课讲清它怎么做、用合成数据验证它能识别真因果方向、在宏观数据上实测, 并**重点澄清它和真正因果的区别**(一个经典面试陷阱)。
> Forecasting cares about **correlation** — predicting is enough. But sometimes we ask deeper: "**does X actually influence/predict Y?**" (Does ad spend drive sales? Do rates affect inflation?) The **Granger causality test** gives an operational answer: if **adding X's history significantly improves predicting Y** (beyond Y's own history alone), then "X **Granger-causes** Y." We explain how it works, validate it detects true causal direction on synthetic data, test it on macro data, and **crucially clarify how it differs from true causation** (a classic interview trap).
>
> 💼 **实战/面试视角**："Granger 因果是什么 / 它和真因果的区别(超高频陷阱) / 怎么检验 / 局限" 是计量/因果岗常考。
> 💼 **Practical/interview angle:** "what Granger causality is / vs true causation (a very common trap) / how to test / limitations" — econometrics/causality roles.

> 📐 **符号约定 / Notation**
> - "X Granger-导致 Y" —— X 的过去能改善对 Y 的预测 / X's past improves predicting Y
> - 原假设 H0:"X 不 Granger-导致 Y" / null: X does not Granger-cause Y

> 💡 **面试相关 / Interview-relevant**
> - "Granger 因果检验的原理"（出镜率 ★★★★）
> - "Granger 因果 ≠ 真正因果(为什么)"（出镜率 ★★★★★，经典陷阱）
> - "怎么做(VAR/受限vs非受限模型)"（★★★）
> - "前提条件(平稳性)"（★★★）

---

## 学习目标 / Learning Objectives
1. 理解 Granger 因果的定义(预测改善)与检验思路。
   Understand Granger causality (prediction improvement) and the test.
2. 用合成数据验证它能识别**真因果方向**。
   Validate on synthetic data that it detects the true direction.
3. 在宏观数据上做 Granger 检验。
   Run a Granger test on macro data.
4. **澄清 Granger 因果 ≠ 真因果**及其局限。
   Clarify Granger causality ≠ true causation and its limits.

## 目录 / TOC
1. [预测 vs 因果 + Granger 定义 ⭐](#1)
2. [合成验证:识别真因果方向 ⭐](#2)
3. [宏观数据实测 ⭐](#3)
4. [Granger ≠ 真因果 + 小结 ⭐](#4)


<a id="1"></a>
## 1. 预测 vs 因果 + Granger 定义 ⭐ / Prediction vs Causation & Granger's Definition

前面所有方法都在做**预测**(相关就够)。但**相关不等于因果**(Part 2 反复强调)——冰淇淋销量和溺水人数高度相关, 但互不导致(共同原因: 夏天)。**因果推断**问的是更难的问题: X 变化会不会**导致** Y 变化?
All prior methods do **forecasting** (correlation suffices). But **correlation ≠ causation** (stressed in Part 2) — ice-cream sales and drownings correlate highly but neither causes the other (common cause: summer). **Causal inference** asks the harder question: would changing X **cause** Y to change?

**Granger 因果(Clive Granger, 诺奖)** 给出一个**基于预测**的、可操作的定义(面试核心)：
**Granger causality (Clive Granger, Nobel laureate)** gives an operational, **prediction-based** definition (interview core):
> 如果**用 Y 自己的过去 + X 的过去**预测 Y, 比**只用 Y 自己的过去**预测得**显著更好**, 就说 "**X Granger-导致 Y**"。
> If predicting Y from **Y's own past + X's past** is **significantly better** than from **Y's own past alone**, then "**X Granger-causes Y**."

直觉: X 的历史里**含有关于 Y 未来的、Y 自己历史里没有的信息**。检验方法: 比较两个模型(都是回归)——
Intuition: X's history contains **information about Y's future not in Y's own history**. The test compares two regression models —
- **受限模型**: 只用 Y 的过去预测 Y。
  **Restricted:** predict Y from Y's past only.
- **非受限模型**: 用 Y 的过去 + X 的过去预测 Y。
  **Unrestricted:** predict Y from Y's past + X's past.
- 若非受限**显著降低预测误差**(F 检验, p<0.05), 则 X Granger-导致 Y。**原假设 H0 = "X 不 Granger-导致 Y"**。
  If the unrestricted model **significantly reduces error** (F-test, p<0.05), X Granger-causes Y. The **null H0 = "X does not Granger-cause Y."**


<a id="2"></a>
## 2. 合成验证:识别真因果方向 ⭐ / Synthetic Validation

先用**人造数据**验证 Granger 检验靠不靠谱: 我们构造 **X 真的影响 Y**(Y 的未来由 X 的过去决定), 但 **Y 不影响 X**。一个好的检验应该: 检出 "**X→Y 显著**", 而 "**Y→X 不显著**"。
First validate on **synthetic data** where **X truly drives Y** (Y's future depends on X's past) but **Y doesn't drive X**. A good test should find "**X→Y significant**" but "**Y→X not significant**."


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings("ignore")
from statsmodels.tsa.stattools import grangercausalitytests
sns.set_theme(style="whitegrid"); np.random.seed(0)

# 构造: X是独立噪声; Y的未来由X的过去决定(X真的导致Y, 滞后2期) / construct: X causes Y with lag 2
n = 500
x = np.random.randn(n)
y = np.zeros(n)
for t in range(2, n):
    y[t] = 0.6 * x[t-2] + 0.3 * y[t-1] + np.random.randn() * 0.5   # Y依赖X的过去 → X导致Y / Y depends on X's past
data = pd.DataFrame({"X": x, "Y": y})

def granger_p(df, cause, effect, maxlag=4):
    # 检验 cause 是否 Granger-导致 effect; 约定: 测第2列是否导致第1列 / test if col2 Granger-causes col1
    res = grangercausalitytests(df[[effect, cause]], maxlag=maxlag, verbose=False)
    return min(res[lag][0]["ssr_ftest"][1] for lag in range(1, maxlag+1))   # 取各滞后最小p值 / min p over lags

p_xy = granger_p(data, cause="X", effect="Y")            # X → Y ? (真因果) / true direction
p_yx = granger_p(data, cause="Y", effect="X")            # Y → X ? (无因果) / no causation
print("合成数据(我们构造的真相: X导致Y, Y不导致X):")
print(f"  X → Y 的 Granger p值 = {p_xy:.4f}  {'→ 显著(检出X导致Y) ✓' if p_xy<0.05 else '→ 不显著'}")
print(f"  Y → X 的 Granger p值 = {p_yx:.4f}  {'→ 显著' if p_yx<0.05 else '→ 不显著(正确, Y不导致X) ✓'}")
print("\nGranger检验正确识别了因果方向: X→Y显著, Y→X不显著 → 检验有效")
print("做法: 比较'只用Y过去'vs'用Y过去+X过去'预测Y, 后者显著更好(F检验)→ X Granger-导致 Y")


<a id="3"></a>
## 3. 宏观数据实测 ⭐ / Real Macro Data

在真实宏观经济数据上做 Granger 检验。一个经典问题: **投资的历史能否帮助预测 GDP?**(投资是否 Granger-导致 GDP) 注意要先**平稳化**(Granger 检验要求平稳, 同 14.4/14.9)。
Run Granger tests on real macro data. A classic question: **does investment's history help predict GDP?** (Does investment Granger-cause GDP?) Note we must **stationarize first** (Granger requires stationarity, like 14.4/14.9).


In [ ]:
import statsmodels.api as sm
macro = sm.datasets.macrodata.load_pandas().data
g = np.log(macro[["realgdp", "realcons", "realinv"]]).diff().dropna()*100   # 增长率(平稳) / growth rates (stationary)

pairs = [("realinv", "realgdp"), ("realcons", "realgdp"), ("realgdp", "realinv")]
print("宏观数据 Granger 因果检验(p<0.05 表示前者有助于预测后者):")
for cause, effect in pairs:
    p = grangercausalitytests(g[[effect, cause]], maxlag=4, verbose=False)
    pmin = min(p[lag][0]["ssr_ftest"][1] for lag in range(1, 5))
    verdict = "Granger-导致 ✓" if pmin < 0.05 else "无显著Granger因果"
    print(f"  {cause:9} → {effect:9}: p={pmin:.4f}  → {verdict}")
print("\n注: 这只说明'前者的历史有助于预测后者'(预测意义上的因果), 不代表真正的经济因果(见下)")


<a id="4"></a>
## 4. Granger ≠ 真因果 + 小结 ⭐ / Granger ≠ True Causation

**最重要的一点(面试超高频陷阱)**:**Granger 因果 ≠ 真正的因果**! 名字里有"因果"极具误导性。Granger 检验只说明 **"X 的过去对预测 Y 有用"**(一种**预测意义上的领先关系**), 并**不证明 X 真的导致 Y**。原因(面试必答)：
**The most important point (a very common interview trap):** **Granger causality ≠ true causation!** The name is misleading. The test only shows **"X's past helps predict Y"** (a **predictive lead-lag relationship**), and does **not prove X truly causes Y**. Why (must-know):
- **共同原因(混淆)**:可能有第三个变量 Z **同时影响** X 和 Y, 只是 Z 先影响 X 再影响 Y, 造成 "X 领先 Y" 的假象。X 并不真的导致 Y。
  **Common cause (confounding):** a third variable Z may **drive both** X and Y, hitting X first then Y, creating an illusory "X leads Y." X doesn't truly cause Y.
- **只是"领先指标"**:X 可能只是 Y 的一个**领先指标**(如"打雷"领先"下雨", 但打雷不导致下雨, 共同原因是低气压)。
  **Just a leading indicator:** X may merely **lead** Y (thunder precedes rain, but thunder doesn't cause rain — common cause: low pressure).
- **遗漏变量、反向因果**:漏掉关键变量或真实方向相反, 都会误导 Granger 结果。
  **Omitted variables, reverse causation:** can all mislead Granger results.

> ⚠️ 所以面试问"Granger 因果能证明因果吗?"——**不能**! 它是基于**预测**和**时间先后**的统计关系, 真正的因果需要**实验(随机对照)或专门的因果推断方法**(Part 后续/反事实)。Granger 适合做**探索性的领先-滞后分析**, 不能下因果结论。
> ⚠️ So if asked "does Granger causality prove causation?" — **No!** It's a statistical relation based on **prediction** and **temporal precedence**; true causation needs **experiments (RCTs) or dedicated causal inference**. Granger is for **exploratory lead-lag analysis**, not causal conclusions.

```
预测 vs 因果: 预测只需相关; 因果问"X变化会否导致Y变化"(相关≠因果)
Granger因果: 若'Y过去+X过去'预测Y 显著优于 '仅Y过去', 则X Granger-导致Y
做法: 比较受限(仅Y过去)vs非受限(+X过去)模型, F检验; H0="X不Granger-导致Y", p<0.05则导致
前提: 序列平稳(先差分); 可双向检验(X→Y 和 Y→X 都测)
⚠️核心陷阱: Granger因果 ≠ 真因果! 只是"预测意义的领先关系"; 可能是共同原因/领先指标/遗漏变量
真因果需要: 实验(RCT)或因果推断方法; Granger只做探索性领先-滞后分析, 不能下因果结论
```

### 💡 面试速查 / Interview cheat-sheet
1. **Granger因果**: X过去能否显著改善对Y的预测(超过Y自身过去); F检验比较两模型。
   Granger causality: does X's past significantly improve predicting Y (beyond Y's own); F-test of two models.
2. **≠真因果(超高频陷阱)**: 只是预测领先关系, 可能是共同原因/领先指标; 不证明因果。
   ≠ true causation: a predictive lead-lag; may be common cause/leading indicator; not proof.
3. **做法**: 受限(仅Y过去) vs 非受限(+X过去), 显著改善则Granger-导致。
   How: restricted vs unrestricted models; significant improvement → Granger-causes.
4. **前提**: 平稳(先差分); 可双向测。
   Prereq: stationarity; can test both directions.
5. **真因果需要**: 实验/因果推断; Granger只做探索性分析。
   True causation needs experiments/causal inference; Granger is exploratory.

### 🎉 Part 14 完成 / Part 14 Complete
你已走完**时间序列**: 从基础(趋势/季节/平稳/ACF)、分解(STL)、平滑(Holt-Winters)、**ARIMA/SARIMA**, 到 Prophet、**LSTM/TCN/Transformer** 深度预测、多变量(VAR)、异常检测、Granger 因果。一条线覆盖了经典统计到现代深度学习的时序预测全谱, 并贯穿强调了**按时间切分防泄漏、相关≠因果、小数据上经典法常胜深度学习**等关键实战认知。
You've completed **Time Series**: from basics (trend/seasonality/stationarity/ACF), decomposition (STL), smoothing (Holt-Winters), **ARIMA/SARIMA**, to Prophet, **LSTM/TCN/Transformer** deep forecasting, multivariate (VAR), anomaly detection, and Granger causality — spanning classical statistics to modern deep learning, while stressing key practical truths: **chronological splits to avoid leakage, correlation ≠ causation, and classical methods often beat deep learning on small data**.
